# AdaptCLIP-Repro: train on VisA

Select **GPU T4 ×2**, enable **Internet**, and **Run All**. No secrets or attached
datasets are required. Public assets and training source are pinned and verified.

The baseline trains fresh adapters for **15 epochs**, seed 10, FP32,
**batch 4/GPU = global batch 8**, DDP + SyncBatchNorm. This differs from the authors'
single-GPU BatchNorm recipe; benchmark evaluation is outside this notebook.
The optional smoke test uses separate outputs and does not initialize the baseline.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

REPO = 'https://github.com/Purin1410/AdaptCLIP-Repro.git'
REVISION = '1d4859b066b8a449b1a40836fce7b762425b9f96'
DOWNLOADS = Path('/kaggle/temp/adaptclip/downloads')
DATA = Path('/kaggle/temp/adaptclip/data/visa')
CHECKOUT = Path('/kaggle/temp/AdaptCLIP-Repro')
OUTPUT = Path('/kaggle/working/adaptclip_visa_seed10')
RUN_SMOKE_TESTS = True  # Optional: validates training and fresh-process resume first.
DOWNLOADS.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
print('Training source:', REVISION)


## 1. Download official VisA, split CSV and pretrained CLIP


In [ ]:
manifest_url = f'https://raw.githubusercontent.com/Purin1410/AdaptCLIP-Repro/{REVISION}/assets.json'
with urllib.request.urlopen(manifest_url, timeout=60) as response:
    assets = json.load(response)['assets']

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

for name, asset in assets.items():
    path = DOWNLOADS / name
    if not path.is_file() or sha256(path) != asset['sha256']:
        print('Downloading', name, flush=True)
        temporary = path.with_suffix(path.suffix + '.partial')
        with urllib.request.urlopen(asset['url'], timeout=120) as source, temporary.open('wb') as target:
            shutil.copyfileobj(source, target, 2 ** 20)
        assert sha256(temporary) == asset['sha256'], f'Hash mismatch: {name}'
        temporary.replace(path)
    print('Verified:', name, flush=True)


## 2. Clone the pinned training code and install dependencies


In [ ]:
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', REPO, str(CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip() == REVISION
assert not subprocess.check_output(['git', '-C', str(CHECKOUT), 'status', '--porcelain'], text=True).strip(), 'Use a clean checkout'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-input', '-r', str(CHECKOUT / 'requirements-kaggle.txt')], check=True)
os.chdir(CHECKOUT)


## 3. Prepare native VisA data and metadata


In [ ]:
subprocess.run([sys.executable, 'scripts/prepare_visa.py',
                '--downloads', str(DOWNLOADS), '--data', str(DATA)], check=True)


## 4. Optional bounded GPU and resume checks

Eight source queries, separate scratch outputs: one single-GPU update and a DDP
run compared with a fresh-process resumed run. This takes a few minutes.
Set `RUN_SMOKE_TESTS = False` to skip it on later runs of an already verified revision.


In [ ]:
if RUN_SMOKE_TESTS:
    subprocess.run([sys.executable, 'tests/gpu_smoke.py', '--data', str(DATA),
                    '--pretrained_dir', str(DOWNLOADS),
                    '--scratch', '/kaggle/temp/adaptclip/smoke',
                    '--out', '/kaggle/working/runtime_validation'], check=True)


## 5. Train the baseline

Fresh adapters, 15 epochs, 271 synchronized updates per epoch. The backbone remains
frozen; all three adapters train. Checkpoints/logs are saved under `OUTPUT`.
`epoch_15.pth` is the final inference checkpoint; `last.pth` also stores optimizer/RNG.

For epoch-boundary recovery, keep the same flags and append
`--resume /kaggle/input/YOUR_CHECKPOINT/last.pth`. `--epoch 15` stays the total target.
Choose a new output directory for a fresh run; existing outputs are never silently replaced.


In [ ]:
!python -u train.py \
    --dataset visa --train_data_path {DATA} \
    --pretrained_dir {DOWNLOADS} --save_path {OUTPUT} \
    --devices 2 --sync_bn --batch_size 4 \
    --epoch 15 --learning_rate 0.001 --seed 10 \
    --image_size 518 --features_list 6 12 18 24 \
    --n_ctx 12 --vl_reduction 4 --pq_mid_dim 128 \
    --visual_learner --textual_learner --pq_learner --pq_context \
    --save_freq 1 --print_freq 1
assert _exit_code == 0, 'Training failed; inspect the error and saved logs above.'
